# Day 1 — Data Foundation
### Enterprise HR AI — Workforce Intelligence & Upskilling Platform

## Project Overview

This notebook lays the **data foundation** for the Workforce Intelligence & Upskilling
platform. Before any modelling happens (that's Day 2), the raw HR data needs to be
understood, checked for problems, cleaned, and mapped out so every table can be joined
correctly.

**What this notebook does, in order:**

| Step | Section | What it answers |
|------|---------|------------------|
| 1 | **Data Understanding** | What data do we have — shape, columns, types, gaps? |
| 2 | **Data Validation** | Where exactly is the data broken, before we touch it? |
| 3 | **Data Cleaning** | Fix everything Section 2 found; save clean copies |
| 4 | **Data Relationships** | How do the 5 tables connect, and is it safe to join them? |

**Ground rule:** no modelling today. This notebook only understands, validates, cleans,
and connects the data — nothing else. Day 2 starts from clean, documented data.

### The five source files

| File | Contents | Feeds into |
|------|----------|------------|
| `employee_attrition.csv` | One row per employee: demographics, role, satisfaction, attrition flag | The attrition ML model (Day 2) |
| `hr_performance_engagement.csv` | Engagement score, performance rating per employee | Engagement analytics |
| `occupation_data.csv` | One row per job role (the "role master" table) | Reference table other data joins to |
| `essential_skills.csv` | Non-software skills required per role | Skills-gap analysis |
| `software_skills.csv` | Software/tool skills required per role | Skills-gap analysis |

---

## 0. Setup

### 0.1 Importing Libraries

Everything this notebook needs, imported up front: `pandas`/`numpy` for data handling,
`os` for the folder structure, and `matplotlib` for the one chart we make (the missing
target balance).

In [1]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt

# Show every column when printing a DataFrame, instead of truncating with '...'
pd.set_option("display.max_columns", None)

# Fixed seed so anything random (synthetic data, sampling) is reproducible
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

### 0.2 Project Folder Structure

The project keeps raw data, cleaned data, and generated docs in separate folders, so the
original files are never overwritten:

| Folder | Purpose |
|--------|---------|
| `data/raw/` | The 5 original CSV files, untouched |
| `data/processed/` | Cleaned copies produced by Section 3 |
| `docs/` | Generated documentation, e.g. `data_relationships.md` from Section 4 |

`os.makedirs(..., exist_ok=True)` creates these folders if they don't exist yet, and does
nothing if they already do.

In [2]:
BASE = "."  # this notebook lives in notebooks/, so ../data is the project's data/ folder
RAW_PATH = f"{BASE}/data/raw"
PROCESSED_PATH = f"{BASE}/data/processed"
DOCS_PATH = f"{BASE}/docs"

for folder in [RAW_PATH, PROCESSED_PATH, DOCS_PATH]:
    os.makedirs(folder, exist_ok=True)

EXPECTED_FILES = [
    "employee_attrition.csv",
    "hr_performance_engagement.csv",
    "occupation_data.csv",
    "essential_skills.csv",
    "software_skills.csv",
]

print("Files currently in data/raw:", os.listdir(RAW_PATH))

Files currently in data/raw: ['Cleaned_HR_Data_Analysis.csv', 'employee_attrition.csv', 'Employee_Performance_Dataset.csv', 'employee_performance_pro.csv', 'essential_skills.csv', 'occupation_data.csv', 'software_skills.csv']


### 0.3 Demo Data Generator

In [3]:
def generate_synthetic_raw_data(raw_path, seed=RANDOM_STATE, n_employees=600):
    """Creates the 5 raw CSVs with realistic messiness, only if the real files
    aren't already in `raw_path`. The 'messiness' (missing values, duplicates,
    impossible values, inconsistent text) is intentional -- it's exactly what
    Sections 2 and 3 below are built to catch and fix.
    """
    rng = np.random.default_rng(seed)

    # ---- 1. employee_attrition.csv ----
    departments = ["Sales", "Research & Development", "Human Resources", "IT"]
    job_roles = ["Sales Executive", "Data Analyst", "ML Engineer", "HR Executive",
                 "Research Scientist", "Software Engineer", "MLOps Engineer"]

    rows = []
    for i in range(1, n_employees + 1):
        dept = rng.choice(departments, p=[0.28, 0.30, 0.10, 0.32])
        # inconsistent casing injected on purpose, ~5% of rows
        if rng.random() < 0.05:
            dept = dept.lower()

        age = int(np.clip(rng.normal(37, 9), 18, 60))
        years_at_company = max(0, int(rng.exponential(5)))
        job_satisfaction = int(rng.integers(1, 5))
        work_life_balance = int(rng.integers(1, 5))
        overtime = rng.choice(["Yes", "No"], p=[0.28, 0.72])
        monthly_income = max(1500, int(rng.normal(6500, 3500)))
        distance_from_home = int(rng.integers(1, 30))
        num_companies_worked = int(rng.integers(0, 8))
        years_since_promotion = int(rng.integers(0, min(years_at_company + 1, 15)))

        # a simple synthetic "risk score" -- higher risk pushes attrition probability up
        risk = (
            0.9 * (5 - job_satisfaction) + 0.7 * (5 - work_life_balance)
            + 1.2 * (overtime == "Yes") + 0.4 * (years_since_promotion > 4)
            + 0.03 * distance_from_home + 0.5 * (years_at_company < 2)
            - 0.00015 * monthly_income + rng.normal(0, 1.2)
        )
        attrition = "Yes" if rng.random() < 1 / (1 + np.exp(-(risk - 2.5))) else "No"

        rows.append(dict(
            EmployeeID=i, Age=age, Department=dept,
            JobRole=rng.choice(job_roles), MonthlyIncome=monthly_income,
            DistanceFromHome=distance_from_home, YearsAtCompany=years_at_company,
            YearsSinceLastPromotion=years_since_promotion,
            NumCompaniesWorked=num_companies_worked,
            JobSatisfaction=job_satisfaction, WorkLifeBalance=work_life_balance,
            OverTime=overtime, Attrition=attrition,
        ))

    attrition_df = pd.DataFrame(rows)

    # inject dirtiness: missing values in a few columns
    for col in ["MonthlyIncome", "JobSatisfaction", "WorkLifeBalance"]:
        missing_idx = rng.choice(attrition_df.index, size=int(0.03 * n_employees), replace=False)
        attrition_df.loc[missing_idx, col] = np.nan

    # inject dirtiness: a handful of duplicate rows
    dup_rows = attrition_df.sample(n=5, random_state=seed)
    attrition_df = pd.concat([attrition_df, dup_rows], ignore_index=True)

    # inject dirtiness: one impossible age, one duplicate EmployeeID
    attrition_df.loc[attrition_df.index[-1], "Age"] = 999
    attrition_df.loc[attrition_df.index[-2], "EmployeeID"] = attrition_df.loc[0, "EmployeeID"]

    attrition_df.to_csv(f"{raw_path}/employee_attrition.csv", index=False)

    # ---- 2. hr_performance_engagement.csv ----
    real_ids = attrition_df["EmployeeID"].unique()
    eng_rows = []
    for eid in real_ids:
        eng_rows.append(dict(
            EmployeeID=eid,
            EngagementScore=int(np.clip(rng.normal(70, 15), 0, 100)),
            PerformanceRating=int(rng.integers(1, 5)),
            ManagerRating=int(rng.integers(1, 6)),
            TrainingHoursLastYear=int(rng.integers(0, 60)),
        ))
    engagement_df = pd.DataFrame(eng_rows)

    # inject dirtiness: a couple of out-of-range engagement scores (>100)
    bad_idx = rng.choice(engagement_df.index, size=3, replace=False)
    engagement_df.loc[bad_idx, "EngagementScore"] = [250, 180, -10]

    # inject dirtiness: a few missing PerformanceRating
    missing_idx = rng.choice(engagement_df.index, size=8, replace=False)
    engagement_df.loc[missing_idx, "PerformanceRating"] = np.nan

    engagement_df.to_csv(f"{raw_path}/hr_performance_engagement.csv", index=False)

    # ---- 3. occupation_data.csv ----
    occupation_df = pd.DataFrame([
        {"OccupationID": 1, "RoleName": "Sales Executive", "Department": "Sales"},
        {"OccupationID": 2, "RoleName": "Data Analyst", "Department": "Research & Development"},
        {"OccupationID": 3, "RoleName": "ML Engineer", "Department": "Research & Development"},
        {"OccupationID": 4, "RoleName": "HR Executive", "Department": "Human Resources"},
        {"OccupationID": 5, "RoleName": "Research Scientist", "Department": "Research & Development"},
        {"OccupationID": 6, "RoleName": "Software Engineer", "Department": "IT"},
        {"OccupationID": 7, "RoleName": "MLOps Engineer", "Department": "IT"},
    ])
    occupation_df.to_csv(f"{raw_path}/occupation_data.csv", index=False)

    # ---- 4. essential_skills.csv ----
    essential_skills = {
        1: ["Negotiation", "CRM Tools", "Communication"],
        2: ["SQL", "Excel", "Statistics", "Data Visualization"],
        3: ["Python", "Machine Learning", "MLOps", "Statistics"],
        4: ["Communication", "HR Compliance", "Conflict Resolution"],
        5: ["Python", "Statistics", "Research Methods"],
        6: ["Python", "Java", "System Design", "Docker"],
        7: ["MLOps", "Docker", "Cloud", "CI/CD"],
    }
    ess_rows = [
        {"OccupationID": oid, "Skill": skill, "Importance": rng.choice(["High", "Medium"])}
        for oid, skills in essential_skills.items() for skill in skills
    ]
    essential_df = pd.DataFrame(ess_rows)
    essential_df.to_csv(f"{raw_path}/essential_skills.csv", index=False)

    # ---- 5. software_skills.csv ----
    # Deliberately inconsistent spelling of the same tool ('AWS' vs 'Amazon Web
    # Services' vs 'AWS Cloud') -- this is exactly what Section 3 (cleaning) fixes.
    software_skills = {
        1: ["Salesforce", "Excel"],
        2: ["SQL", "Tableau", "Excel"],
        3: ["AWS", "Docker", "PyTorch"],
        4: ["Workday", "Excel"],
        5: ["Python (Jupyter)", "AWS Cloud"],
        6: ["Docker", "Kubernetes", "AWS"],
        7: ["Amazon Web Services", "Docker", "Kubernetes", "Terraform"],
    }
    sw_rows = [
        {"OccupationID": oid, "SoftwareSkill": skill, "Importance": rng.choice(["High", "Medium"])}
        for oid, skills in software_skills.items() for skill in skills
    ]
    software_df = pd.DataFrame(sw_rows)
    software_df.to_csv(f"{raw_path}/software_skills.csv", index=False)

    print("Synthetic raw data generated in", raw_path)


if not all(os.path.exists(f"{RAW_PATH}/{f}") for f in EXPECTED_FILES):
    generate_synthetic_raw_data(RAW_PATH)
else:
    print("Real files already present in data/raw -- using those, not generating synthetic data.")

print("Files now in data/raw:", os.listdir(RAW_PATH))

Synthetic raw data generated in ./data/raw
Files now in data/raw: ['Cleaned_HR_Data_Analysis.csv', 'employee_attrition.csv', 'Employee_Performance_Dataset.csv', 'employee_performance_pro.csv', 'essential_skills.csv', 'hr_performance_engagement.csv', 'occupation_data.csv', 'software_skills.csv']


## 1. Data Understanding

For each of the five datasets: shape, columns, dtypes, missing values, duplicates, and
any obvious ID column to join on.

### 1.1 Loading the Raw Files

All five CSVs are loaded into one dictionary, `raw`, keyed by a short name. Keeping
them in a dictionary (instead of 5 separate variables) makes it easy to loop over all
of them in the profiling and validation steps below.

In [4]:
raw_files = {
    "employee_attrition": "employee_attrition.csv",
    "hr_performance_engagement": "hr_performance_engagement.csv",
    "occupation_data": "occupation_data.csv",
    "essential_skills": "essential_skills.csv",
    "software_skills": "software_skills.csv",
}

raw = {name: pd.read_csv(f"{RAW_PATH}/{fname}") for name, fname in raw_files.items()}
list(raw.keys())

['employee_attrition',
 'hr_performance_engagement',
 'occupation_data',
 'essential_skills',
 'software_skills']

### 1.2 Profiling Every Dataset

`profile_dataset` prints the same set of diagnostics for any DataFrame: shape, column
types, top missing-value columns, duplicate row count, and any column with "id" in its
name (a likely join key). 

In [5]:
def profile_dataset(name, df):
    print(f"{'='*60}")
    print(f"{name}")
    print(f"{'='*60}")
    print(f"Shape: {df.shape}")

    print("\nColumns and dtypes:")
    print(df.dtypes)

    print("\nMissing values (top 20):")
    print(df.isnull().sum().sort_values(ascending=False).head(20))

    print(f"\nDuplicate rows: {df.duplicated().sum()}")

    id_cols = [c for c in df.columns if "id" in c.lower()]
    print(f"\nColumns with 'id' in the name (possible join keys): {id_cols}")
    print()


for name, df in raw.items():
    profile_dataset(name, df)

employee_attrition
Shape: (605, 13)

Columns and dtypes:
EmployeeID                   int64
Age                          int64
Department                  object
JobRole                     object
MonthlyIncome              float64
DistanceFromHome             int64
YearsAtCompany               int64
YearsSinceLastPromotion      int64
NumCompaniesWorked           int64
JobSatisfaction            float64
WorkLifeBalance            float64
OverTime                    object
Attrition                   object
dtype: object

Missing values (top 20):
MonthlyIncome              18
JobSatisfaction            18
WorkLifeBalance            18
EmployeeID                  0
Age                         0
JobRole                     0
Department                  0
DistanceFromHome            0
YearsAtCompany              0
NumCompaniesWorked          0
YearsSinceLastPromotion     0
OverTime                    0
Attrition                   0
dtype: int64

Duplicate rows: 3

Columns with 'id' in the 

### 1.3 A Closer Look at Each File

**`employee_attrition.csv`** — the main file, one row per employee.

In [6]:
raw["employee_attrition"].head()

,EmployeeID,Age,Department,JobRole,MonthlyIncome,DistanceFromHome,YearsAtCompany,YearsSinceLastPromotion,NumCompaniesWorked,JobSatisfaction,WorkLifeBalance,OverTime,Attrition
0,1,43,IT,Sales Executive,6947.0,21,1,1,6,1.0,1.0,No,Yes
1,2,47,IT,Research Scientist,NaN,25,6,1,6,2.0,1.0,No,Yes
2,3,33,IT,Software Engineer,7944.0,22,2,2,5,2.0,2.0,Yes,Yes
3,4,47,Research & Development,Software Engineer,8777.0,28,1,0,3,3.0,2.0,Yes,Yes
4,5,38,IT,Software Engineer,8876.0,24,1,1,0,1.0,3.0,Yes,Yes


In [7]:
raw["employee_attrition"].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 605 entries, 0 to 604
Data columns (total 13 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   EmployeeID               605 non-null    int64  
 1   Age                      605 non-null    int64  
 2   Department               605 non-null    object 
 3   JobRole                  605 non-null    object 
 4   MonthlyIncome            587 non-null    float64
 5   DistanceFromHome         605 non-null    int64  
 6   YearsAtCompany           605 non-null    int64  
 7   YearsSinceLastPromotion  605 non-null    int64  
 8   NumCompaniesWorked       605 non-null    int64  
 9   JobSatisfaction          587 non-null    float64
 10  WorkLifeBalance          587 non-null    float64
 11  OverTime                 605 non-null    object 
 12  Attrition                605 non-null    object 
dtypes: float64(3), int64(6), object(4)
memory usage: 61.6+ KB


Since `Attrition` is the target column the Day 2 model will predict, it's worth
checking the class balance now — a very imbalanced target (say, 95% "No") changes how
that model needs to be built.

In [8]:
# Attrition file specifically -- check the target balance
raw["employee_attrition"]["Attrition"].value_counts(normalize=True) * 100

Attrition
Yes    74.710744
No     25.289256
Name: proportion, dtype: float64

**`hr_performance_engagement.csv`** — engagement and performance, one row per employee.

In [9]:
raw["hr_performance_engagement"].head()

,EmployeeID,EngagementScore,PerformanceRating,ManagerRating,TrainingHoursLastYear
0,1,81,3.0,3,28
1,2,61,2.0,1,8
2,3,60,4.0,5,38
3,4,59,1.0,3,39
4,5,67,2.0,3,2


**`occupation_data.csv`** — the role master table other files will join to.

In [10]:
raw["occupation_data"].head()

,OccupationID,RoleName,Department
0,1,Sales Executive,Sales
1,2,Data Analyst,Research & Development
2,3,ML Engineer,Research & Development
3,4,HR Executive,Human Resources
4,5,Research Scientist,Research & Development


**`essential_skills.csv`** — non-software skills required per role.

In [11]:
raw["essential_skills"].head()

,OccupationID,Skill,Importance
0,1,Negotiation,High
1,1,CRM Tools,Medium
2,1,Communication,Medium
3,2,SQL,Medium
4,2,Excel,High


**`software_skills.csv`** — software/tool skills required per role. Note the inconsistent spelling of the same tools (`AWS` / `AWS Cloud` / `Amazon Web Services`) — flagged now, fixed in Section 3.

In [12]:
raw["software_skills"].head()

,OccupationID,SoftwareSkill,Importance
0,1,Salesforce,High
1,1,Excel,Medium
2,2,SQL,Medium
3,2,Tableau,Medium
4,2,Excel,High


### 1.4 What Each Dataset Is Actually For

| Dataset | Role in the project |
|---------|----------------------|
| `employee_attrition.csv` | Feeds the attrition ML model directly (Day 2) |
| `hr_performance_engagement.csv` | Engagement analytics — no ML needed at first |
| `occupation_data.csv` | Becomes the "role master" reference table |
| `essential_skills.csv` + `software_skills.csv` | Combine into one required-skills table per role |

### 1.5 End-of-Section-1 Summary

One row per dataset — row/column count, likely primary key, how much is missing, and
how many duplicate rows exist. This table is the input to Section 4 (Data
Relationships), and gets printed again in the final Day 1 summary.

In [13]:
summary_rows = []
for name, df in raw.items():
    id_cols = [c for c in df.columns if "id" in c.lower()]
    missing_pct = round(100 * df.isnull().sum().sum() / (df.shape[0] * df.shape[1]), 2)
    summary_rows.append({
        "dataset": name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "likely_primary_key": id_cols[0] if id_cols else "(none obvious)",
        "missing_pct": missing_pct,
        "duplicate_rows": df.duplicated().sum(),
    })

understanding_summary = pd.DataFrame(summary_rows)
understanding_summary

,dataset,rows,columns,likely_primary_key,missing_pct,duplicate_rows
0,employee_attrition,605,13,EmployeeID,0.69,3
1,hr_performance_engagement,600,5,EmployeeID,0.27,0
2,occupation_data,7,3,OccupationID,0.00,0
3,essential_skills,25,3,OccupationID,0.00,0
4,software_skills,19,3,OccupationID,0.00,0


---

## 2. Data Validation

Defines what "valid" means for this data *before* cleaning it, so bad rows are caught
loudly instead of quietly turning into bad predictions later. The checks below cover
schema (right columns present), type (numbers are numbers), range (values make sense),
uniqueness (no duplicate IDs), and category (only expected values appear).

> **Note:** nothing gets merged/joined yet, even where two files clearly share an ID.
> That happens in Section 4, only after the relationship is actually confirmed — not
> assumed from a matching column name.

### 2.1 A Reusable Check Function

`check()` is a soft assertion: instead of stopping the notebook at the first failure
(what a plain `assert` would do), it records every failure in `validation_issues` and
keeps going — so Section 2 surfaces *every* problem in one pass, not just the first one.

In [14]:
validation_issues = {name: [] for name in raw}

def check(name, condition, message):
    """Runs one validation check. Records a failure instead of crashing the
    notebook, so all issues are visible at once rather than stopping at the first.
    """
    if not condition:
        validation_issues[name].append(message)
        print(f"[FAIL] {name}: {message}")
    else:
        print(f"[ OK ] {name}: {message}")

### 2.2 Validating `employee_attrition.csv`

Checks: all expected columns exist, `Age`/`MonthlyIncome` are numeric, `Age` is a
realistic value, `EmployeeID` is unique, `Attrition` only contains `Yes`/`No`, and
`Department` only contains known department names.

In [15]:
df = raw["employee_attrition"]

expected_cols = {"EmployeeID", "Age", "Department", "JobRole", "MonthlyIncome",
                  "YearsAtCompany", "OverTime", "JobSatisfaction", "WorkLifeBalance",
                  "Attrition"}
check("employee_attrition", expected_cols.issubset(df.columns),
      f"expected columns present (missing: {expected_cols - set(df.columns)})")

check("employee_attrition", pd.api.types.is_numeric_dtype(df["Age"]),
      "Age is numeric")
check("employee_attrition", pd.api.types.is_numeric_dtype(df["MonthlyIncome"]),
      "MonthlyIncome is numeric")

check("employee_attrition", df["Age"].dropna().between(18, 100).all(),
      "Age is between 18 and 100")
check("employee_attrition", df["EmployeeID"].is_unique,
      "EmployeeID has no duplicates")
check("employee_attrition", set(df["Attrition"].dropna().unique()) <= {"Yes", "No"},
      "Attrition only contains Yes/No")
check("employee_attrition",
      set(df["Department"].dropna().str.strip().str.title().unique()) <=
      {"Sales", "Research & Development", "Human Resources", "It", "Information Technology"},
      "Department values look like known departments (case-insensitive)")

[ OK ] employee_attrition: expected columns present (missing: set())
[ OK ] employee_attrition: Age is numeric
[ OK ] employee_attrition: MonthlyIncome is numeric
[FAIL] employee_attrition: Age is between 18 and 100
[FAIL] employee_attrition: EmployeeID has no duplicates
[ OK ] employee_attrition: Attrition only contains Yes/No
[ OK ] employee_attrition: Department values look like known departments (case-insensitive)


### 2.3 Validating `hr_performance_engagement.csv`

Checks: expected columns exist, `EmployeeID` is unique, and `EngagementScore` is a
valid percentage (0–100).

In [16]:
df = raw["hr_performance_engagement"]

expected_cols = {"EmployeeID", "EngagementScore", "PerformanceRating"}
check("hr_performance_engagement", expected_cols.issubset(df.columns),
      f"expected columns present (missing: {expected_cols - set(df.columns)})")

check("hr_performance_engagement", df["EmployeeID"].is_unique,
      "EmployeeID has no duplicates")
check("hr_performance_engagement", df["EngagementScore"].dropna().between(0, 100).all(),
      "EngagementScore is between 0 and 100")

[ OK ] hr_performance_engagement: expected columns present (missing: set())
[ OK ] hr_performance_engagement: EmployeeID has no duplicates
[FAIL] hr_performance_engagement: EngagementScore is between 0 and 100


### 2.4 Validating `occupation_data.csv`

This is the smallest, cleanest file — checks: `OccupationID` is unique, `RoleName`
has no missing values.

In [17]:
df = raw["occupation_data"]

check("occupation_data", df["OccupationID"].is_unique, "OccupationID has no duplicates")
check("occupation_data", df["RoleName"].notnull().all(), "RoleName has no missing values")

[ OK ] occupation_data: OccupationID has no duplicates
[ OK ] occupation_data: RoleName has no missing values


### 2.5 Validating `essential_skills.csv` and `software_skills.csv`

Both files should have an `OccupationID` column, and every `OccupationID` in them
should actually exist in `occupation_data` — an "orphan" ID here means a skill is
linked to a role that doesn't exist, which would silently drop rows on a later join.

In [18]:
for name in ["essential_skills", "software_skills"]:
    df = raw[name]
    check(name, "OccupationID" in df.columns, "has an OccupationID column to join on")
    valid_occ_ids = set(raw["occupation_data"]["OccupationID"])
    check(name, set(df["OccupationID"].unique()).issubset(valid_occ_ids),
          "every OccupationID exists in occupation_data (no orphan foreign keys)")

[ OK ] essential_skills: has an OccupationID column to join on
[ OK ] essential_skills: every OccupationID exists in occupation_data (no orphan foreign keys)
[ OK ] software_skills: has an OccupationID column to join on
[ OK ] software_skills: every OccupationID exists in occupation_data (no orphan foreign keys)


### 2.6 Validation Summary

Every issue found above, grouped by dataset. `total_issues` is what Section 3 needs to
fix — nothing gets silently dropped without first being logged here.

In [19]:
print("VALIDATION SUMMARY\n" + "="*60)
total_issues = sum(len(v) for v in validation_issues.values())
for name, issues in validation_issues.items():
    status = "CLEAN" if not issues else f"{len(issues)} ISSUE(S)"
    print(f"  {name}: {status}")
    for issue in issues:
        print(f"      - {issue}")
print(f"\nTotal issues found: {total_issues}")
print("These get fixed in Section 3 (Data Cleaning) below --")
print("nothing gets silently dropped without being logged first.")

VALIDATION SUMMARY
  employee_attrition: 2 ISSUE(S)
      - Age is between 18 and 100
      - EmployeeID has no duplicates
  hr_performance_engagement: 1 ISSUE(S)
      - EngagementScore is between 0 and 100
  occupation_data: CLEAN
  essential_skills: CLEAN
  software_skills: CLEAN

Total issues found: 3
These get fixed in Section 3 (Data Cleaning) below --
nothing gets silently dropped without being logged first.


---

## 3. Data Cleaning

Fixes everything Section 2 flagged: missing values, duplicates, wrong types,
inconsistent categories, out-of-range values, and messy skill-name spelling
(`'AWS'` vs `'Amazon Web Services'` vs `'AWS Cloud'` all meaning the same thing).

Every cleaning step below is logged with `log_clean()`, and the output is a **clean
copy** of each file — the originals in `raw` are never modified, and the cleaned files
are saved separately in `data/processed/` at the end of this section.

### 3.1 A Reusable Logging Function

Same idea as `check()` in Section 2: one small helper, `log_clean()`, records every
cleaning action (and how many rows/values it touched) so there's a full audit trail of
what changed and why.

In [20]:
cleaning_log = []

def log_clean(dataset, action, count=None):
    """Records one cleaning action to cleaning_log and prints it immediately."""
    entry = f"{dataset}: {action}" + (f" ({count} rows/values affected)" if count is not None else "")
    cleaning_log.append(entry)
    print(entry)

### 3.2 Cleaning `employee_attrition.csv`

Five separate problems, fixed one at a time so each fix (and its row count) is
logged individually rather than buried in one large block.

**Step 1 — remove exact duplicate rows.**

In [21]:
df = raw["employee_attrition"].copy()

before = len(df)
df = df.drop_duplicates()
log_clean("employee_attrition", "dropped exact duplicate rows", before - len(df))

employee_attrition: dropped exact duplicate rows (3 rows/values affected)


**Step 2 — remove rows with a duplicate `EmployeeID`** (keeping the first occurrence). A duplicate ID means two rows claim to be the same employee, which would corrupt any join on this key.

In [22]:
before = len(df)
df = df.drop_duplicates(subset="EmployeeID", keep="first")
log_clean("employee_attrition", "dropped rows with duplicate EmployeeID (kept first)", before - len(df))

employee_attrition: dropped rows with duplicate EmployeeID (kept first) (2 rows/values affected)


**Step 3 — fix impossible `Age` values.** Section 2 requires `Age` between 18 and 100; anything outside that range is treated as missing (not guessed at), then filled in Step 5 below.

In [23]:
bad_age = ~df["Age"].between(18, 100)
log_clean("employee_attrition", "impossible Age values found, set to missing", int(bad_age.sum()))
df.loc[bad_age, "Age"] = np.nan  # treat impossible ages as missing, not guessed

employee_attrition: impossible Age values found, set to missing (0 rows/values affected)


**Step 4 — standardize `Department` casing.** `.str.title()` turns `'sales'` into `'Sales'`; the one exception is `'It'`, which needs to become `'IT'` explicitly.

In [24]:
df["Department"] = df["Department"].str.strip().str.title()
df["Department"] = df["Department"].replace({"It": "IT"})
log_clean("employee_attrition", "standardized Department casing (e.g. 'sales' -> 'Sales')")

employee_attrition: standardized Department casing (e.g. 'sales' -> 'Sales')


**Step 5 — fill missing numeric values with the column median.** Median is used (not mean) because these are small integer scales (1–4 satisfaction scores, income) where the median is less sensitive to outliers.

In [25]:
for col in ["MonthlyIncome", "JobSatisfaction", "WorkLifeBalance", "Age"]:
    n_missing = df[col].isnull().sum()
    df[col] = df[col].fillna(df[col].median())
    log_clean("employee_attrition", f"filled missing {col} with median", int(n_missing))

attrition_clean = df
attrition_clean.head()

employee_attrition: filled missing MonthlyIncome with median (18 rows/values affected)
employee_attrition: filled missing JobSatisfaction with median (18 rows/values affected)
employee_attrition: filled missing WorkLifeBalance with median (18 rows/values affected)
employee_attrition: filled missing Age with median (0 rows/values affected)


,EmployeeID,Age,Department,JobRole,MonthlyIncome,DistanceFromHome,YearsAtCompany,YearsSinceLastPromotion,NumCompaniesWorked,JobSatisfaction,WorkLifeBalance,OverTime,Attrition
0,1,43.0,IT,Sales Executive,6947.0,21,1,1,6,1.0,1.0,No,Yes
1,2,47.0,IT,Research Scientist,6285.5,25,6,1,6,2.0,1.0,No,Yes
2,3,33.0,IT,Software Engineer,7944.0,22,2,2,5,2.0,2.0,Yes,Yes
3,4,47.0,Research & Development,Software Engineer,8777.0,28,1,0,3,3.0,2.0,Yes,Yes
4,5,38.0,IT,Software Engineer,8876.0,24,1,1,0,1.0,3.0,Yes,Yes


### 3.3 Cleaning `hr_performance_engagement.csv`

Three problems: duplicate `EmployeeID` rows, `EngagementScore` values outside the
valid 0–100 range, and missing `PerformanceRating`.

In [26]:
df = raw["hr_performance_engagement"].copy()

before = len(df)
df = df.drop_duplicates(subset="EmployeeID", keep="first")
log_clean("hr_performance_engagement", "dropped duplicate EmployeeID rows", before - len(df))

out_of_range = ~df["EngagementScore"].between(0, 100)
log_clean("hr_performance_engagement", "EngagementScore values outside 0-100 clipped", int(out_of_range.sum()))
df["EngagementScore"] = df["EngagementScore"].clip(lower=0, upper=100)

n_missing_perf = df["PerformanceRating"].isnull().sum()
df["PerformanceRating"] = df["PerformanceRating"].fillna(df["PerformanceRating"].median())
log_clean("hr_performance_engagement", "filled missing PerformanceRating with median", int(n_missing_perf))

engagement_clean = df
engagement_clean.head()

hr_performance_engagement: dropped duplicate EmployeeID rows (0 rows/values affected)
hr_performance_engagement: EngagementScore values outside 0-100 clipped (3 rows/values affected)
hr_performance_engagement: filled missing PerformanceRating with median (8 rows/values affected)


,EmployeeID,EngagementScore,PerformanceRating,ManagerRating,TrainingHoursLastYear
0,1,81,3.0,3,28
1,2,61,2.0,1,8
2,3,60,4.0,5,38
3,4,59,1.0,3,39
4,5,67,2.0,3,2


### 3.4 Cleaning `occupation_data.csv`

This file was already the cleanest of the five (Section 2 found no issues) — the only
work needed is standardizing whitespace and casing on the text columns, for
consistency with the other cleaned files.

In [27]:
df = raw["occupation_data"].copy()
df["RoleName"] = df["RoleName"].str.strip()
df["Department"] = df["Department"].str.strip().str.title().replace({"It": "IT"})

occupation_clean = df
log_clean("occupation_data", "standardized RoleName/Department whitespace and casing")
occupation_clean

occupation_data: standardized RoleName/Department whitespace and casing


,OccupationID,RoleName,Department
0,1,Sales Executive,Sales
1,2,Data Analyst,Research & Development
2,3,ML Engineer,Research & Development
3,4,HR Executive,Human Resources
4,5,Research Scientist,Research & Development
5,6,Software Engineer,IT
6,7,MLOps Engineer,IT


### 3.5 Cleaning `essential_skills.csv` and `software_skills.csv`

This is the problem called out in Section 1.3: the same tool spelled several
different ways (`'AWS'`, `'AWS Cloud'`, `'Amazon Web Services'`). A small canonical
mapping — lowercased spelling → correct display name — fixes all of them at once. In a
real project this dictionary grows as new synonyms are found; keeping it in one place
(instead of scattered `.replace()` calls) makes it easy to extend.

**Step 1 — define the canonical skill-name mapping and a helper to apply it.**

In [28]:
SKILL_CANONICAL_MAP = {
    "aws": "AWS",
    "aws cloud": "AWS",
    "amazon web services": "AWS",
    "python (jupyter)": "Python",
    "python": "Python",
    "sql": "SQL",
    "docker": "Docker",
    "kubernetes": "Kubernetes",
    "pytorch": "PyTorch",
    "excel": "Excel",
    "tableau": "Tableau",
    "salesforce": "Salesforce",
    "workday": "Workday",
    "terraform": "Terraform",
}

def standardize_skill(raw_skill):
    """Looks up a skill name (case-insensitive) in the canonical map. Anything
    not in the map is left as-is, just with whitespace trimmed."""
    key = raw_skill.strip().lower()
    return SKILL_CANONICAL_MAP.get(key, raw_skill.strip())

**Step 2 — apply it to `essential_skills.csv`.**

In [29]:
essential_clean = raw["essential_skills"].copy()
essential_clean["Skill"] = essential_clean["Skill"].apply(standardize_skill)
log_clean("essential_skills", "standardized skill name spelling via canonical map")
essential_clean.head()

essential_skills: standardized skill name spelling via canonical map


,OccupationID,Skill,Importance
0,1,Negotiation,High
1,1,CRM Tools,Medium
2,1,Communication,Medium
3,2,SQL,Medium
4,2,Excel,High


**Step 3 — apply it to `software_skills.csv`**, then drop any duplicate rows the standardization created (e.g. a role listing both `'AWS'` and `'AWS Cloud'` becomes two identical `'AWS'` rows).

In [30]:
software_clean = raw["software_skills"].copy()

before_unique = software_clean["SoftwareSkill"].nunique()
software_clean["SoftwareSkill"] = software_clean["SoftwareSkill"].apply(standardize_skill)
after_unique = software_clean["SoftwareSkill"].nunique()
log_clean("software_skills",
          f"standardized skill name spelling via canonical map (unique values: {before_unique} -> {after_unique})")

before = len(software_clean)
software_clean = software_clean.drop_duplicates()
log_clean("software_skills", "dropped duplicate rows created by standardization", before - len(software_clean))

software_clean

software_skills: standardized skill name spelling via canonical map (unique values: 13 -> 11)
software_skills: dropped duplicate rows created by standardization (0 rows/values affected)


,OccupationID,SoftwareSkill,Importance
0,1,Salesforce,High
1,1,Excel,Medium
2,2,SQL,Medium
3,2,Tableau,Medium
4,2,Excel,High
5,3,AWS,Medium
6,3,Docker,High
7,3,PyTorch,Medium
8,4,Workday,High
9,4,Excel,Medium


### 3.6 Cleaning Log Summary

Every action taken above, in one place — this is the audit trail for what changed
between `raw` and the `*_clean` DataFrames.

In [31]:
print("CLEANING LOG SUMMARY\n" + "="*60)
for entry in cleaning_log:
    print(f"  - {entry}")

CLEANING LOG SUMMARY
  - employee_attrition: dropped exact duplicate rows (3 rows/values affected)
  - employee_attrition: dropped rows with duplicate EmployeeID (kept first) (2 rows/values affected)
  - employee_attrition: impossible Age values found, set to missing (0 rows/values affected)
  - employee_attrition: standardized Department casing (e.g. 'sales' -> 'Sales')
  - employee_attrition: filled missing MonthlyIncome with median (18 rows/values affected)
  - employee_attrition: filled missing JobSatisfaction with median (18 rows/values affected)
  - employee_attrition: filled missing WorkLifeBalance with median (18 rows/values affected)
  - employee_attrition: filled missing Age with median (0 rows/values affected)
  - hr_performance_engagement: dropped duplicate EmployeeID rows (0 rows/values affected)
  - hr_performance_engagement: EngagementScore values outside 0-100 clipped (3 rows/values affected)
  - hr_performance_engagement: filled missing PerformanceRating with median (8

### 3.7 Re-Validation After Cleaning

Re-runs the key Section 2 checks against the cleaned data, to *prove* the cleaning
actually worked rather than just assuming it did. If any check still fails, the
`assert` below stops the notebook before saving bad data.

In [32]:
print("RE-VALIDATION AFTER CLEANING\n" + "="*60)

recheck = {
    "employee_attrition: no duplicate EmployeeID": attrition_clean["EmployeeID"].is_unique,
    "employee_attrition: Age in range": attrition_clean["Age"].between(18, 100).all(),
    "employee_attrition: no missing MonthlyIncome": attrition_clean["MonthlyIncome"].isnull().sum() == 0,
    "hr_performance_engagement: EngagementScore in range": engagement_clean["EngagementScore"].between(0, 100).all(),
    "hr_performance_engagement: no duplicate EmployeeID": engagement_clean["EmployeeID"].is_unique,
}

for check_name, passed in recheck.items():
    print(f"  [{'OK' if passed else 'FAIL'}] {check_name}")

assert all(recheck.values()), "Some post-cleaning checks still fail -- fix before saving processed data."
print("\nAll checks pass -- safe to save processed files.")

RE-VALIDATION AFTER CLEANING
  [OK] employee_attrition: no duplicate EmployeeID
  [OK] employee_attrition: Age in range
  [OK] employee_attrition: no missing MonthlyIncome
  [OK] hr_performance_engagement: EngagementScore in range
  [OK] hr_performance_engagement: no duplicate EmployeeID

All checks pass -- safe to save processed files.


### 3.8 Saving the Cleaned Files

The output of Day 1: five cleaned CSVs written to `data/processed/`, kept separate
from `data/raw/` so the original files are never overwritten.

In [33]:
attrition_clean.to_csv(f"{PROCESSED_PATH}/employee_attrition_processed.csv", index=False)
engagement_clean.to_csv(f"{PROCESSED_PATH}/engagement_processed.csv", index=False)
occupation_clean.to_csv(f"{PROCESSED_PATH}/occupation_master.csv", index=False)
essential_clean.to_csv(f"{PROCESSED_PATH}/essential_skills_processed.csv", index=False)
software_clean.to_csv(f"{PROCESSED_PATH}/software_skills_processed.csv", index=False)

print("Processed files written to", PROCESSED_PATH)
print(os.listdir(PROCESSED_PATH))

Processed files written to ./data/processed
['career_path_intelligence.csv', 'employee_attrition_processed.csv', 'employee_intelligence.csv', 'engagement_processed.csv', 'essential_skills_processed.csv', 'occupation_master.csv', 'org_skill_heatmap.csv', 'software_skills_processed.csv']


---

## 4. Data Relationships

Deciding how the five cleaned tables actually connect, confirming each join key really
works, and writing it all down in `docs/data_relationships.md` so the mapping doesn't
live only in someone's head.

### 4.1 The Join Key Table

For every pair of tables: the join key, the relationship type (one-to-one,
one-to-many, many-to-one), and why.

In [34]:
relationships = [
    {
        "table_a": "employee_attrition",
        "table_b": "hr_performance_engagement",
        "join_key": "EmployeeID",
        "relationship": "one-to-one",
        "reason": "Same employee's performance/engagement record.",
    },
    {
        "table_a": "employee_attrition",
        "table_b": "occupation_data",
        "join_key": "JobRole (attrition) <-> RoleName (occupation)",
        "relationship": "many-to-one",
        "reason": (
            "Many employees share the same JobRole. Note: this is a TEXT join, not an "
            "ID join -- JobRole and RoleName must match exactly after cleaning, which is "
            "riskier than an ID join and worth double-checking (see verification below)."
        ),
    },
    {
        "table_a": "occupation_data",
        "table_b": "essential_skills",
        "join_key": "OccupationID",
        "relationship": "one-to-many",
        "reason": "Each role requires several essential (non-software) skills.",
    },
    {
        "table_a": "occupation_data",
        "table_b": "software_skills",
        "join_key": "OccupationID",
        "relationship": "one-to-many",
        "reason": "Each role requires several software/tool skills.",
    },
]

relationships_df = pd.DataFrame(relationships)
relationships_df

,table_a,table_b,join_key,relationship,reason
0,employee_attrition,hr_performance_engagement,EmployeeID,one-to-one,Same employee's performance/engagement record.
1,employee_attrition,occupation_data,JobRole (attrition) <-> RoleName (occupation),many-to-one,Many employees share the same JobRole. Note: t...
2,occupation_data,essential_skills,OccupationID,one-to-many,Each role requires several essential (non-soft...
3,occupation_data,software_skills,OccupationID,one-to-many,Each role requires several software/tool skills.


### 4.2 Verifying the Risky Join: `JobRole` ↔ `RoleName`

Every other join key above is an ID, which is safe by construction. This one is a
**text** join — two columns have to match on spelling exactly, which is much easier to
get subtly wrong. Before relying on it anywhere, check that every `JobRole` value in
the cleaned attrition data actually has a matching `RoleName` in the occupation table.

In [35]:
attrition_roles = set(attrition_clean["JobRole"].unique())
occupation_roles = set(occupation_clean["RoleName"].unique())

unmatched_in_attrition = attrition_roles - occupation_roles
matched = attrition_roles & occupation_roles

print(f"JobRole values in employee_attrition: {len(attrition_roles)}")
print(f"Matched against occupation_data.RoleName: {len(matched)}")
print(f"Unmatched (would silently fail to join): {unmatched_in_attrition if unmatched_in_attrition else 'none'}")

assert not unmatched_in_attrition, (
    "Some JobRole values don't exist in occupation_data -- fix spelling/casing before "
    "joining on this key in Day 3."
)
print("\nAll JobRole values have a matching RoleName -- safe to join on this text key.")

JobRole values in employee_attrition: 7
Matched against occupation_data.RoleName: 7
Unmatched (would silently fail to join): none

All JobRole values have a matching RoleName -- safe to join on this text key.


### 4.3 Verifying the ID Join: `EmployeeID`

A quick sanity check that `EmployeeID` in both files really does refer to the same set
of employees — a matching column name alone isn't proof of a matching key.

In [36]:
common_ids = set(attrition_clean["EmployeeID"]) & set(engagement_clean["EmployeeID"])

print(f"EmployeeIDs in employee_attrition: {attrition_clean['EmployeeID'].nunique()}")
print(f"EmployeeIDs in hr_performance_engagement: {engagement_clean['EmployeeID'].nunique()}")
print(f"EmployeeIDs present in both: {len(common_ids)}")

EmployeeIDs in employee_attrition: 600
EmployeeIDs in hr_performance_engagement: 600
EmployeeIDs present in both: 600


### 4.4 Entity Diagram

A simple text diagram of how everything connects, for a quick visual reference.

In [37]:
entity_diagram = """
EMPLOYEE
  |
  +-- Employee ID ---- Engagement Data      (one-to-one, via EmployeeID)
  |
  +-- Job Role ------- Occupation Data      (many-to-one, via JobRole <-> RoleName)
        |
        +-- Essential Skills                 (one-to-many, via OccupationID)
        +-- Software Skills                  (one-to-many, via OccupationID)
"""
print(entity_diagram)


EMPLOYEE
  |
  +-- Employee ID ---- Engagement Data      (one-to-one, via EmployeeID)
  |
  +-- Job Role ------- Occupation Data      (many-to-one, via JobRole <-> RoleName)
        |
        +-- Essential Skills                 (one-to-many, via OccupationID)
        +-- Software Skills                  (one-to-many, via OccupationID)



### 4.5 Writing `docs/data_relationships.md`

Everything above — the diagram, the join key table, and the verification results —
combined into one markdown file, so the next person (or Day 2) doesn't have to re-derive
any of it from the notebook.

In [38]:
doc_lines = []
doc_lines.append("# Data Relationships\n")
doc_lines.append(
    "How the five raw tables connect. Written after actually confirming each key "
    "lines up -- not assumed from matching column names.\n"
)
doc_lines.append("## Entity diagram\n")
doc_lines.append("```")
doc_lines.append(entity_diagram.strip())
doc_lines.append("```\n")
doc_lines.append("## Join key table\n")
doc_lines.append(relationships_df.to_markdown(index=False))
doc_lines.append("")
doc_lines.append("## Verification notes\n")
doc_lines.append(
    f"- `EmployeeID`: {len(common_ids)} IDs confirmed present in both "
    "`employee_attrition` and `hr_performance_engagement`.\n"
    f"- `JobRole` <-> `RoleName`: all {len(attrition_roles)} distinct JobRole values in "
    "`employee_attrition` have a matching `RoleName` in `occupation_data` after "
    "cleaning (Section 3 standardized casing on both).\n"
)

with open(f"{DOCS_PATH}/data_relationships.md", "w") as f:
    f.write("\n".join(doc_lines))

print(f"Written to {DOCS_PATH}/data_relationships.md")

Written to ./docs/data_relationships.md


In [39]:
with open(f"{DOCS_PATH}/data_relationships.md") as f:
    print(f.read())

# Data Relationships

How the five raw tables connect. Written after actually confirming each key lines up -- not assumed from matching column names.

## Entity diagram

```
EMPLOYEE
  |
  +-- Employee ID ---- Engagement Data      (one-to-one, via EmployeeID)
  |
  +-- Job Role ------- Occupation Data      (many-to-one, via JobRole <-> RoleName)
        |
        +-- Essential Skills                 (one-to-many, via OccupationID)
        +-- Software Skills                  (one-to-many, via OccupationID)
```

## Join key table

| table_a            | table_b                   | join_key                                      | relationship   | reason                                                                                                                                                                                                                           |
|:-------------------|:--------------------------|:----------------------------------------------|:---------------|:-----

---

## End of Day 1 — Summary

What's in hand at the end of today, per the project checklist.

In [40]:
print("DAY 1 COMPLETE\n" + "="*60)

print("\n1. Data Understanding -- profiled all 5 raw datasets")
print(understanding_summary.to_string(index=False))

print(f"\n2. Data Validation -- {total_issues} issue(s) found and logged before cleaning")

print(f"\n3. Data Cleaning -- {len(cleaning_log)} cleaning actions applied, all post-cleaning checks pass")
print(f"   Processed files saved to: {PROCESSED_PATH}")
for f in os.listdir(PROCESSED_PATH):
    print(f"     - {f}")

print(f"\n4. Data Relationships -- {len(relationships_df)} table relationships confirmed and documented")
print(f"   Written to: {DOCS_PATH}/data_relationships.md")

print("\nReady for Day 2 (Machine Learning) -- feature engineering starts from")
print(f"   {PROCESSED_PATH}/employee_attrition_processed.csv")

DAY 1 COMPLETE

1. Data Understanding -- profiled all 5 raw datasets
                  dataset  rows  columns likely_primary_key  missing_pct  duplicate_rows
       employee_attrition   605       13         EmployeeID         0.69               3
hr_performance_engagement   600        5         EmployeeID         0.27               0
          occupation_data     7        3       OccupationID         0.00               0
         essential_skills    25        3       OccupationID         0.00               0
          software_skills    19        3       OccupationID         0.00               0

2. Data Validation -- 3 issue(s) found and logged before cleaning

3. Data Cleaning -- 15 cleaning actions applied, all post-cleaning checks pass
   Processed files saved to: ./data/processed
     - career_path_intelligence.csv
     - employee_attrition_processed.csv
     - employee_intelligence.csv
     - engagement_processed.csv
     - essential_skills_processed.csv
     - occupation_master.